[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab06_speech_processing.ipynb)

# ปฏิบัติการที่ 6 (Lab 6): การประมวลผลเสียงพูดสำหรับงานด้านสุขภาพ (Speech Processing for Healthcare)

**รายวิชา:** 3099704 AI for Digital Health (2026) · **วันที่เรียน:** วันพฤหัสบดีที่ 15 ตุลาคม 2026 · **เวลาโดยประมาณ:** 75 นาที

## วัตถุประสงค์การเรียนรู้

- ถอดความเสียงพูดทางการแพทย์ด้วยโมเดลรู้จำเสียงพูดแบบเปิด (OpenAI Whisper) ซึ่งรันบน Colab ได้โดยไม่มีค่าใช้จ่ายและไม่ต้องใช้ API key
- ฟังคลิปเสียงของผู้ป่วยจริง และเปรียบเทียบ Whisper ขนาดเล็กกับขนาดใหญ่ในด้านความแม่นยำและความเร็ว
- วัดคุณภาพการถอดความด้วยอัตราความผิดพลาดระดับคำ (Word Error Rate, WER) และอธิบายได้ว่าเหตุใดค่า WER ที่ต่ำจึงยังอาจมีความผิดพลาดที่เป็นอันตรายแฝงอยู่
- สร้างกระบวนการประมวลผล (pipeline) ขนาดเล็กสำหรับแปลงบันทึกเสียง (voice note) เป็นสรุปแบบมีโครงสร้าง

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (คลิกปุ่ม ▶ หรือกด **Shift+Enter**) ในแต่ละ ⭐ **แบบฝึกหัด (Exercise)** ผู้เรียนเพียงเปลี่ยนค่าหนึ่งค่าหรือเติมโค้ดสั้น ๆ หนึ่งบรรทัด และมีเซลล์ ✅ **เฉลย** อยู่ถัดไป ให้ทดลองทำด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำส่วนถัดไป จากนั้นให้ตอบคำถาม ✍️ ในเซลล์ข้อความที่กำหนด **คำตอบดังกล่าวคือชิ้นงานที่ต้องส่ง** ควรใช้หูฟัง เนื่องจากการฟังคลิปเสียงเป็นส่วนหนึ่งของปฏิบัติการ
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** ส่วนท้ายของปฏิบัติการเป็นการถอดความคลิปเสียงผู้ป่วย 30 คลิปและส่งผลไปยังตารางอันดับ (leaderboard) ของรายวิชา ส่วนนี้ไม่บังคับและไม่นับเป็นชิ้นงานที่ต้องส่ง
>
> หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom

> **Runtime (แนะนำให้ใช้ GPU):** ก่อนรันเซลล์ใด ให้เลือก `Runtime → Change runtime type → T4 GPU → Save` GPU รุ่น T4 ซึ่งใช้ได้โดยไม่มีค่าใช้จ่ายจะทำให้ Whisper ทำงานได้เร็วขึ้นอย่างมาก ปฏิบัติการนี้สามารถรันบน CPU ได้เช่นกัน แต่จะช้ากว่า (การเปรียบเทียบขนาดโมเดลจะใช้เวลาหลายนาทีแทนที่จะเป็นไม่กี่วินาที)

> 🏆 ตารางอันดับซึ่งไม่บังคับนี้ให้คะแนนโดยอัตโนมัติบน [Hugging Face leaderboard](https://huggingface.co/spaces/wdittaya/aidh-leaderboard) ของรายวิชา ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์การตั้งค่าด้านล่างเฉพาะกรณีที่ต้องการเข้าร่วม (ผู้สอนจะประกาศรหัสเข้าร่วมในการเรียนการสอนผ่าน Zoom) อย่างไรก็ตาม ผู้เรียนทุกคนต้องรันเซลล์การตั้งค่าก่อน


In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
!pip -q install gradio_client huggingface_hub

HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
LAB_ID    = "lab06"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

from huggingface_hub import hf_hub_download

def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    from gradio_client import Client, handle_file
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    client = Client(SPACE_ID, verbose=False)
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## ส่วนที่ 1 — การรู้จำเสียงพูดอัตโนมัติด้วย Whisper

การบันทึกเวชระเบียนเป็นภาระงานสำคัญของบุคลากรทางการแพทย์ **การรู้จำเสียงพูดอัตโนมัติ (automatic speech recognition, ASR)** คือระบบที่แปลงเสียงพูดเป็นข้อความ เช่น ระบบผู้ช่วยบันทึกทางการแพทย์ด้วยปัญญาประดิษฐ์ (AI medical scribe) ซึ่งเป็นการประยุกต์ใช้ที่สำคัญด้านสุขภาพดิจิทัล

ปฏิบัติการนี้ใช้ **Whisper** ซึ่งเป็นโมเดลเสียงพูดของ OpenAI ที่เปิดเผยค่าน้ำหนักของโมเดล (**open weights**) ไฟล์โมเดลจะถูกดาวน์โหลดมาประมวลผลภายใน Colab ของผู้เรียนเอง จึงไม่มีการส่งไฟล์เสียงไปยังบริการภายนอก และไม่ต้องใช้ API key Whisper มีหลายขนาดดังนี้

| ขนาด | จำนวนพารามิเตอร์ | ความเร็ว | ความแม่นยำ |
|---|---|---|---|
| `tiny` | 39 M | เร็วที่สุด | ต่ำที่สุด |
| `base` | 74 M | เร็ว | ปานกลาง |
| `small` | 244 M | ปานกลาง | ดีขึ้น |
| `medium` / `turbo` / `large` | 769 M – 1.5 B | ช้าบน Colab รุ่นไม่มีค่าใช้จ่าย | ดีที่สุด |

เซลล์ถัดไปจะติดตั้ง Whisper และโหลดโมเดลขนาด `base` (ใช้เวลาประมาณ 1 นาที)

In [ ]:
!pip -q install openai-whisper jiwer
import re, time, json
import pandas as pd
import torch, whisper, jiwer
from IPython.display import Audio, display

USE_FP16 = torch.cuda.is_available()      # การคำนวณแบบ half precision ใช้ได้เฉพาะบน GPU
print("พบ GPU:", USE_FP16, "" if USE_FP16 else "(ใช้ CPU: ทำงานได้ แต่ช้ากว่า)")
model = whisper.load_model("base")        # ขนาดเหมาะสมกับ Colab รุ่นไม่มีค่าใช้จ่าย

# --- ฟังก์ชันช่วยที่ใช้ตลอดปฏิบัติการ ------------------------------------------------
def normalise(text):
    """แปลงเป็นตัวพิมพ์เล็กและตัดเครื่องหมายวรรคตอน - วิธีเดียวกับที่ตารางอันดับใช้ก่อนให้คะแนน"""
    text = str(text).lower().replace("-", " ").replace("/", " ")
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

def wer_report(reference, hypothesis, label=""):
    """แสดงค่า WER ของข้อความที่ถอดได้ (hypothesis) เทียบกับข้อความที่ถูกต้อง (reference)"""
    ref, hyp = normalise(reference), normalise(hypothesis)
    out = jiwer.process_words(ref, hyp if hyp else "<empty>")
    print(f"{label}WER = {out.wer:.1%}  (คำที่ผิด {out.substitutions}, คำที่ขาด {out.deletions}, "
          f"คำที่เกิน {out.insertions} คำ; ข้อความอ้างอิงมี {len(ref.split())} คำ)")
    return out.wer

def listen(path):
    """แสดงเครื่องเล่นเสียงสำหรับคลิป"""
    display(Audio(filename=path))

print("โหลดโมเดล Whisper 'base' เรียบร้อยแล้ว")


💡 **คำอธิบายผลลัพธ์** Colab ดาวน์โหลดโปรแกรม Whisper และค่าน้ำหนักของโมเดล `base` (ประมาณ 140 MB) มาไว้ใน session ของผู้เรียน และได้สร้างฟังก์ชันช่วย 3 ฟังก์ชัน ได้แก่ `listen()` สำหรับแสดงเครื่องเล่นเสียง `normalise()` สำหรับแปลงตัวพิมพ์และตัดเครื่องหมายวรรคตอน และ `wer_report()` สำหรับให้คะแนนข้อความที่ถอดได้ (อธิบายในหัวข้อถัดไป)

In [ ]:
# คลิปตัวอย่างสาธารณะ (สุนทรพจน์รับตำแหน่งของ J.F. Kennedy ซึ่งรวมอยู่ในไฟล์ทดสอบของ Whisper)
!wget -q -O sample.flac https://github.com/openai/whisper/raw/main/tests/jfk.flac
listen("sample.flac")
result = model.transcribe("sample.flac", fp16=USE_FP16, language="en")
print("ข้อความที่ Whisper ถอดได้:", result["text"])


### การวัดคุณภาพด้วย WER

**อัตราความผิดพลาดระดับคำ (Word Error Rate, WER)** เปรียบเทียบข้อความที่ระบบถอดได้ (*hypothesis*) กับข้อความที่มนุษย์ถอดไว้อย่างถูกต้อง (*reference* หรือข้อความอ้างอิง)

$$\text{WER} = \frac{\text{คำที่ถูกแทนผิด (substitution)} + \text{คำที่ขาด (deletion)} + \text{คำที่เกิน (insertion)}}{\text{จำนวนคำในข้อความอ้างอิง}}$$

ตัวอย่าง: ข้อความอ้างอิง *"no known drug allergies"* (4 คำ) เทียบกับข้อความที่ถอดได้ *"known drug allergy"* มีคำที่ขาด 1 คำ ("no") และคำที่ถูกแทนผิด 1 คำ ("allergies → allergy") รวมผิด 2 คำจาก 4 คำ คิดเป็น **50 %**

**ค่ายิ่งต่ำยิ่งดี** (0 % หมายถึงถูกต้องทั้งหมด) โดยไม่นับความแตกต่างของตัวพิมพ์ใหญ่-เล็กและเครื่องหมายวรรคตอน การนำไปใช้ทางคลินิกโดยทั่วไปต้องการค่า WER ต่ำกว่า 10 % อย่างชัดเจนสำหรับเสียงพูดในบริบททางการแพทย์

In [ ]:
reference = ("And so my fellow Americans ask not what your country can do for you "
             "ask what you can do for your country")
wer_report(reference, result["text"]);


💡 **คำอธิบายผลลัพธ์** สำหรับเสียงพูดที่ชัดเจนและบันทึกด้วยคุณภาพดี Whisper มักถอดความได้ถูกต้องทั้งหมดหรือเกือบทั้งหมด (WER ใกล้ 0 %) แต่เสียงในสถานการณ์ทางคลินิกจริงซึ่งมีสำเนียง เสียงรบกวน และศัพท์ทางการแพทย์นั้นยากกว่ามาก ส่วนถัดไปจึงใช้เสียงของผู้ป่วยจริง

## ส่วนที่ 2 — การฟังคลิปเสียงของผู้ป่วยจริง

รายวิชาจัดเตรียม **คลิปเสียงสั้น 30 คลิป (ความยาว 3–10 วินาที) ของผู้ที่บรรยายอาการของตนเป็นภาษาอังกฤษ** เช่น *"my knee hurts when I climb stairs"* จากชุดข้อมูล (dataset) สาธารณะ *Medical Speech, Transcription, and Intent* ผู้พูดมีสำเนียงและใช้ไมโครโฟนที่หลากหลาย เซลล์ถัดไปจะดาวน์โหลดรายการคลิปและเล่นคลิปแรก

In [ ]:
manifest = pd.read_csv(lab_data("asr/manifest.csv"))    # คอลัมน์: id, file, seconds

def clip_path(clip_id):
    """ดาวน์โหลดคลิป 1 คลิป (เช่น 'C07') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    return lab_data(f"asr/clips/{clip_id}.wav")

print("จำนวนคลิป", len(manifest), "คลิป ความยาวรวม", round(manifest["seconds"].sum()), "วินาที")
display(manifest.head())
listen(clip_path("C01"))


In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: เลือกคลิปผู้ป่วย ฟัง และให้ Whisper ถอดความ
# หน้าที่ของเซลล์: เล่นคลิปที่เลือก และแสดงข้อความที่โมเดล Whisper 'base' ถอดได้
# สิ่งที่ต้องแก้ไข: แทน ... ด้วยรหัสคลิประหว่าง "C01" ถึง "C30" ภายในเครื่องหมายอัญประกาศ
# คำใบ้:  clip_id = "C17"
clip_id = ...      # TODO

listen(clip_path(clip_id))
clip_text = model.transcribe(clip_path(clip_id), fp16=USE_FP16, language="en")["text"].strip()
print(f"ข้อความที่ Whisper (base) ถอดได้จาก {clip_id}:", clip_text)


In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
clip_id = "C17"

listen(clip_path(clip_id))
clip_text = model.transcribe(clip_path(clip_id), fp16=USE_FP16, language="en")["text"].strip()
print(f"ข้อความที่ Whisper (base) ถอดได้จาก {clip_id}:", clip_text)


💡 **คำอธิบายผลลัพธ์** Whisper แปลงเสียงเป็นข้อความภายในเวลาประมาณหนึ่งถึงสองวินาที ให้เล่นคลิปอีกครั้ง (คลิกปุ่ม ▶ บนเครื่องเล่น) และเปรียบเทียบกับข้อความที่แสดงทีละคำ ผู้เรียนสามารถทดลองกับคลิปอื่นได้โดยเปลี่ยนค่า `clip_id` แล้วรันเซลล์ใหม่

✍️ **คำถามที่ 1** โปรดเขียนข้อความที่ผู้เรียนได้ยินจากคลิปให้ตรงที่สุด และพิจารณาว่า Whisper ถอดความได้ถูกต้องทุกคำหรือไม่ หากไม่ถูกต้อง โปรดระบุคำที่ผิด และอธิบายสาเหตุที่เป็นไปได้ (เช่น สำเนียง ความเร็วในการพูด เสียงรบกวน หรือศัพท์ทางการแพทย์ที่พบไม่บ่อย)

✍️ **คำตอบ:**



In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: เปรียบเทียบ Whisper ขนาดต่าง ๆ บนคลิปชุดเดียวกัน
# หน้าที่ของเซลล์: ถอดความ 6 คลิป (คลิปที่เลือก และอีก 5 คลิปที่มีศัพท์ทางการแพทย์) ด้วยโมเดลแต่ละขนาด พร้อมจับเวลา
# สิ่งที่ต้องแก้ไข: แทน ... ด้วยขนาดโมเดลที่ต้องการเปรียบเทียบ ภายในเครื่องหมายอัญประกาศ คั่นด้วยจุลภาค
# คำใบ้:  sizes_to_compare = ["tiny", "base", "small"]
#        (ไม่ควรใช้ "medium"/"large" บน Colab รุ่นไม่มีค่าใช้จ่าย เนื่องจากทำงานช้าและใช้หน่วยความจำจำนวนมาก)
sizes_to_compare = [...]      # TODO

compare_ids = sorted({clip_id, "C05", "C12", "C17", "C25", "C28"})
models = {"base": model}
table = pd.DataFrame({"clip": compare_ids})
for size in sizes_to_compare:
    if size not in models:
        print(f"กำลังโหลดโมเดล '{size}' ...")
        models[size] = whisper.load_model(size)
    start = time.time()
    table[size] = [models[size].transcribe(clip_path(c), fp16=USE_FP16, language="en")["text"].strip()
                   for c in compare_ids]
    print(f"{size:>6}: {(time.time() - start) / len(compare_ids):.1f} วินาทีต่อคลิป")

pd.set_option("display.max_colwidth", None)
display(table)


In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
sizes_to_compare = ["tiny", "base", "small"]

compare_ids = sorted({clip_id, "C05", "C12", "C17", "C25", "C28"})
models = {"base": model}
table = pd.DataFrame({"clip": compare_ids})
for size in sizes_to_compare:
    if size not in models:
        print(f"กำลังโหลดโมเดล '{size}' ...")
        models[size] = whisper.load_model(size)
    start = time.time()
    table[size] = [models[size].transcribe(clip_path(c), fp16=USE_FP16, language="en")["text"].strip()
                   for c in compare_ids]
    print(f"{size:>6}: {(time.time() - start) / len(compare_ids):.1f} วินาทีต่อคลิป")

pd.set_option("display.max_colwidth", None)
display(table)


💡 **คำอธิบายผลลัพธ์** แต่ละคอลัมน์คือข้อความที่โมเดลแต่ละขนาดถอดจากคลิปเดียวกัน โมเดลที่มีขนาดใหญ่กว่าได้เรียนรู้รูปแบบของเสียงพูดมากกว่า จึงมักรองรับสำเนียงและศัพท์ทางการแพทย์ (*phlegm*, *anemia*, *cruciate ligament*) ได้ดีกว่า แต่ทำงานช้ากว่าและใช้หน่วยความจำมากกว่า ให้ใช้เครื่องเล่นเสียงในเซลล์ถัดไปเพื่อตรวจสอบว่าโมเดลใดถอดความได้ถูกต้อง

In [ ]:
# เครื่องเล่นเสียงของคลิปที่ใช้เปรียบเทียบ - ให้ฟังและพิจารณาว่าคอลัมน์ใดใกล้เคียงกับเสียงพูดมากที่สุด
for c in compare_ids:
    print(c); listen(clip_path(c))


In [ ]:
# @title 🎧 ไม่บังคับ: ผู้เรียนเป็นผู้ถอดความ — พิมพ์ข้อความที่ได้ยินจากคลิปที่เลือก (เป็นภาษาอังกฤษ) แล้วรันเซลล์นี้
my_reference = ""  # @param {type:"string"}

if not my_reference.strip():
    print(f"โปรดพิมพ์ข้อความที่ได้ยินจาก {clip_id} ในช่อง my_reference (หรือภายในเครื่องหมายอัญประกาศ) แล้วรันเซลล์ใหม่ เพื่อแสดงค่า WER ของแต่ละโมเดล")
else:
    for size in sizes_to_compare:
        hyp = table.loc[table["clip"] == clip_id, size].iloc[0]
        wer_report(my_reference, hyp, label=f"{size:>6}: ")


💡 **คำอธิบายผลลัพธ์** การคำนวณ WER จำเป็นต้องมีข้อความที่มนุษย์ถอดไว้อย่างถูกต้อง ในกรณีนี้ผู้เรียนทำหน้าที่เป็นมาตรฐานอ้างอิง (gold standard) ในโครงการจริง แพทย์ พยาบาล หรือผู้ถอดความที่ผ่านการฝึกอบรมจะจัดทำข้อความอ้างอิงสำหรับชุดทดสอบก่อนที่เครื่องมือ medical scribe จะได้รับการอนุมัติให้ใช้งาน

✍️ **คำถามที่ 2** จากผลลัพธ์ข้างต้น โมเดลขนาดใดถอดความคลิปชุดนี้ได้ดีที่สุด และทำงานช้ากว่า `tiny` เพียงใด หากโรงพยาบาลต้องการใช้ระบบรู้จำเสียงพูดบนคอมพิวเตอร์ทั่วไปที่ไม่มี GPU (เพื่อไม่ให้ไฟล์เสียงของผู้ป่วยออกนอกโรงพยาบาล) ผู้เรียนจะเลือกโมเดลขนาดใด และต้องยอมรับข้อจำกัดใดบ้าง จงอธิบาย

✍️ **คำตอบ:**



## ส่วนที่ 3 — ข้อจำกัดของ WER: ความผิดพลาดที่เป็นอันตรายทางคลินิก

WER นับคำที่ผิดทุกคำด้วยน้ำหนักเท่ากัน การขาดคำว่า *"the"* และการขาดคำว่า *"no"* ต่างนับเป็นความผิดพลาด 1 คำ แต่มีเพียงกรณีหลังที่อาจเป็นอันตรายต่อผู้ป่วย ความผิดพลาดของ ASR ที่เป็นอันตรายและพบบ่อย ได้แก่ **การปฏิเสธ (negation)** (*"no allergies"* → *"allergies"*) **ตัวเลขและหน่วย** (*"15"* ↔ *"50"*, *"mg"* ↔ *"mcg"*) **ชื่อยาที่ออกเสียงคล้ายกัน (sound-alike drugs)** (*hydroxyzine* ↔ *hydralazine*) และ **ข้างซ้าย/ขวา**

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: สร้างความผิดพลาด "เพียงเล็กน้อย" ที่เป็นอันตรายทางคลินิก
# หน้าที่ของเซลล์: เปรียบเทียบข้อความถอดความที่ผิดพลาด 2 แบบของคำบอกบันทึก (dictation) เดียวกัน
#                 ด้วยค่า WER และด้วยการตรวจสอบคำสำคัญทางคลินิก
# สิ่งที่ต้องแก้ไข: แทน ... ด้วยสำเนาของ dictation ที่เปลี่ยนหรือลบคำเพียง 1 คำ
#                 จนข้อความมีความหมายที่เป็นอันตรายต่อผู้ป่วย
# คำใบ้:  dangerous_hyp = "Patient has known drug allergies. Start metoprolol 25 milligrams twice daily."
#        (ตัวอย่างอื่น: 25 -> 250, twice -> three times)
dictation     = "Patient has no known drug allergies. Start metoprolol 25 milligrams twice daily."
harmless_hyp  = "The patient has no known drug allergies, start metoprolol 25 milligrams twice a day."
dangerous_hyp = ...      # TODO

critical_words = ["no", "metoprolol", "25", "milligrams", "twice"]   # คำสำคัญที่บุคลากรทางการแพทย์ต้องได้รับอย่างถูกต้อง

def missing_critical(hypothesis):
    hyp_words = normalise(hypothesis).split()
    return [w for w in critical_words if w not in hyp_words]

for label, hyp in [("harmless ", harmless_hyp), ("dangerous", dangerous_hyp)]:
    wer_report(dictation, hyp, label=f"{label}: ")
    print("           คำสำคัญที่ขาดหายไป:", missing_critical(hyp) or "ไม่มี")


In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
dictation     = "Patient has no known drug allergies. Start metoprolol 25 milligrams twice daily."
harmless_hyp  = "The patient has no known drug allergies, start metoprolol 25 milligrams twice a day."
dangerous_hyp = "Patient has known drug allergies. Start metoprolol 25 milligrams twice daily."

critical_words = ["no", "metoprolol", "25", "milligrams", "twice"]   # คำสำคัญที่บุคลากรทางการแพทย์ต้องได้รับอย่างถูกต้อง

def missing_critical(hypothesis):
    hyp_words = normalise(hypothesis).split()
    return [w for w in critical_words if w not in hyp_words]

for label, hyp in [("harmless ", harmless_hyp), ("dangerous", dangerous_hyp)]:
    wer_report(dictation, hyp, label=f"{label}: ")
    print("           คำสำคัญที่ขาดหายไป:", missing_critical(hyp) or "ไม่มี")


💡 **คำอธิบายผลลัพธ์** ข้อความที่ไม่เป็นอันตราย (harmless) มีคำผิด *มากกว่า* (มีคำว่า "the" เกิน และ "daily" → "a day") จึงมีค่า WER *สูงกว่า* ข้อความที่เป็นอันตราย (dangerous) ทั้งที่ข้อความที่เป็นอันตรายกลับความหมายของประวัติการแพ้ยาเป็นตรงกันข้าม ค่า WER เพียงค่าเดียวจึงไม่สามารถสะท้อนความเสี่ยงนี้ได้ การประเมินทางคลินิกจำเป็นต้องตรวจสอบคำสำคัญด้วย ได้แก่ การปฏิเสธ ชื่อยา ขนาดยา หน่วย และข้างซ้าย/ขวา

✍️ **คำถามที่ 3** จากข้อความถอดความของคลิปผู้ป่วยในส่วนที่ 2 มีโมเดลใดถอดความผิดจนความหมายทางคลินิกเปลี่ยนไปหรือไม่ (เช่น ระบุอวัยวะผิด ระบุข้างผิด หรือคำว่า "not" ขาดหายไป) จากนั้นจงอธิบายใน 2–3 ประโยคว่า ความเสี่ยงที่สำคัญที่สุดของความผิดพลาดของ ASR ในการบันทึกเวชระเบียนคืออะไร และเพราะเหตุใดค่า WER เฉลี่ยเพียงอย่างเดียวจึงไม่เพียงพอสำหรับการอนุมัติเครื่องมือ medical scribe

✍️ **คำตอบ:**



## ส่วนที่ 4 — การแปลงบันทึกเสียง (voice note) เป็นสรุปแบบมีโครงสร้าง

เครื่องมือ medical scribe จะมีประโยชน์มากขึ้นเมื่อสามารถแปลงคำพูดอิสระเป็น **ข้อมูลแบบมีโครงสร้าง (structured fields)** เช่น รายการยา สัญญาณชีพ และประวัติการแพ้ยา ซึ่งบันทึกลงเวชระเบียนได้ ส่วนนี้ใช้ **กฎ (rules)** อย่างง่าย โดย *นิพจน์ปกติ (regular expression, regex)* คือรูปแบบของข้อความ เช่น รูปแบบ "คำหนึ่งคำ ตามด้วยตัวเลข และหน่วย" จะค้นพบข้อความ *"aspirin 81 milligrams"*

บันทึกเสียงด้านล่าง (ภาษาอังกฤษ) เป็นตัวอย่างข้อความที่ Whisper ถอดจากการบอกบันทึกของพยาบาล

In [ ]:
# ⭐ Exercise 4 — แบบฝึกหัดที่ 4: สกัดชื่อยาและขนาดยาจากบันทึกเสียง
# หน้าที่ของเซลล์: ค้นหารูปแบบ "ชื่อยา ตัวเลข หน่วย" (รวมถึงความดันโลหิต อาการที่ผู้ป่วยปฏิเสธ และประวัติการแพ้ยา)
#                 และแสดงสรุปแบบมีโครงสร้าง
# สิ่งที่ต้องแก้ไข: แทน ... ด้วยหน่วยของขนาดยาที่ต้องการค้นหา ภายในเครื่องหมายอัญประกาศ คั่นด้วย |  (หมายถึง "หรือ")
#                 โดยให้คำที่สะกดยาวกว่าอยู่ก่อน
# คำใบ้:  DOSE_UNITS = "milligrams|milligram|mg|micrograms|mcg|units"
voice_note = ("Patient is a 67 year old woman with dizziness for two days. No chest pain. "
              "Blood pressure 150 over 95. Continue amlodipine 5 mg once daily and start "
              "aspirin 81 milligrams daily. Allergic to penicillin.")

DOSE_UNITS = ...      # TODO

def structure_note(text):
    drug_dose = re.findall(r"\b([a-z]+)\s+(\d+(?:\.\d+)?)\s*(" + DOSE_UNITS + r")\b", text, flags=re.I)
    bp = re.search(r"(\d{2,3})\s*(?:over|/)\s*(\d{2,3})", text, flags=re.I)
    return {
        "medications": [{"drug": d.lower(), "dose": f"{n} {u.lower()}"} for d, n, u in drug_dose],
        "blood_pressure": f"{bp.group(1)}/{bp.group(2)}" if bp else None,
        "denied_symptoms": re.findall(r"\bno ([a-z ]+?)(?=[.,]|$)", text, flags=re.I),
        "allergies": re.findall(r"allergic to ([a-z]+)", text, flags=re.I),
    }

summary = structure_note(voice_note)
print(json.dumps(summary, indent=2))


In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
voice_note = ("Patient is a 67 year old woman with dizziness for two days. No chest pain. "
              "Blood pressure 150 over 95. Continue amlodipine 5 mg once daily and start "
              "aspirin 81 milligrams daily. Allergic to penicillin.")

DOSE_UNITS = "milligrams|milligram|mg|micrograms|mcg|units"

def structure_note(text):
    drug_dose = re.findall(r"\b([a-z]+)\s+(\d+(?:\.\d+)?)\s*(" + DOSE_UNITS + r")\b", text, flags=re.I)
    bp = re.search(r"(\d{2,3})\s*(?:over|/)\s*(\d{2,3})", text, flags=re.I)
    return {
        "medications": [{"drug": d.lower(), "dose": f"{n} {u.lower()}"} for d, n, u in drug_dose],
        "blood_pressure": f"{bp.group(1)}/{bp.group(2)}" if bp else None,
        "denied_symptoms": re.findall(r"\bno ([a-z ]+?)(?=[.,]|$)", text, flags=re.I),
        "allergies": re.findall(r"allergic to ([a-z]+)", text, flags=re.I),
    }

summary = structure_note(voice_note)
print(json.dumps(summary, indent=2))


💡 **คำอธิบายผลลัพธ์** กฎที่กำหนดค้นพบยาทั้งสองรายการพร้อมขนาดยา ความดันโลหิต อาการที่ผู้ป่วยปฏิเสธ ("no chest pain") และประวัติการแพ้ยา กฎลักษณะนี้โปร่งใสและคาดการณ์ผลได้ แต่ไม่ยืดหยุ่น หากขนาดยาถูกพูดเป็นคำ เช่น *"five milligrams"* หรือชื่อยาถูกถอดความผิด ระบบจะค้นไม่พบหรือบันทึกข้อมูลที่ผิดลงไปโดยตรง เซลล์ถัดไปทดลองใช้กฎชุดเดิมกับข้อความที่มีความผิดพลาดของ ASR

In [ ]:
# บันทึกเดิมที่ Whisper ถอดความผิด: "No chest pain" -> "Now chest pain", "5 mg" -> "50 mg"
misheard = voice_note.replace("No chest pain", "Now chest pain").replace("5 mg", "50 mg")
print(json.dumps(structure_note(misheard), indent=2))


💡 **คำอธิบายผลลัพธ์** pipeline สร้างบันทึกแบบมีโครงสร้างได้ตามปกติ ทั้งที่ **ขนาดยาเพิ่มขึ้นเป็นสิบเท่า** และการปฏิเสธ "no chest pain" หายไปโดยไม่มีการแจ้งเตือน ความผิดพลาดของการรู้จำเสียงพูดจึงส่งผ่านเข้าสู่เวชระเบียนแบบมีโครงสร้างโดยตรง ด้วยเหตุนี้บุคลากรทางการแพทย์จึงต้องตรวจทานและลงนามรับรองบันทึกทุกฉบับที่ปัญญาประดิษฐ์ร่างขึ้น

In [ ]:
# @title 🎙️ ส่วนเสริม (ไม่บังคับ): บอกบันทึกทางคลินิกด้วยเสียงของผู้เรียน (เลือกช่องทำเครื่องหมาย รันเซลล์ แล้วอัปโหลดไฟล์เสียง)
use_my_recording = False  # @param {type:"boolean"}
# บันทึกเสียงประโยคสั้น ๆ ภาษาอังกฤษด้วยโทรศัพท์หรือคอมพิวเตอร์ (wav/mp3/m4a) เช่น
#   "Patient presents with acute chest pain and shortness of breath, prescribe aspirin 81 milligrams."
# ใช้ข้อมูลสมมติเท่านั้น - ห้ามอัปโหลดเสียงของผู้ป่วยจริง

if use_my_recording:
    from google.colab import files
    uploaded = files.upload()                 # เลือกไฟล์เสียง
    audio_path = next(iter(uploaded))
    listen(audio_path)
    my_text = model.transcribe(audio_path, fp16=USE_FP16, language="en")["text"]
    print("ข้อความที่ Whisper ถอดได้:", my_text)
    print(json.dumps(structure_note(my_text), indent=2))
else:
    print("ข้ามส่วนนี้ (เลือก use_my_recording หากต้องการทดลองกับเสียงของผู้เรียนเอง)")


✍️ **คำถามที่ 4** สมมติว่าโรงพยาบาลต้องการนำ AI medical scribe ที่ฟังการตรวจรักษาและกรอกข้อมูลรายการยาและประวัติการแพ้ยาโดยอัตโนมัติมาใช้ โปรดระบุ **มาตรการป้องกันความเสี่ยง (safeguards) อย่างน้อย 2 ข้อ** ที่ควรกำหนดทั้งก่อนและหลังเริ่มใช้งานจริง พร้อมเหตุผล (ประเด็นที่ควรพิจารณา ได้แก่ การตรวจทานโดยบุคลากร การทดสอบกับสำเนียงและชื่อยาที่ใช้ในพื้นที่ ความยินยอมของผู้ป่วย และสถานที่ประมวลผลและจัดเก็บไฟล์เสียง)

✍️ **คำตอบ:**



## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

**ส่วนนี้ไม่บังคับ และไม่นับเป็นชิ้นงานที่ต้องส่ง** ส่วนนี้ถอดความ **คลิปผู้ป่วยทั้ง 30 คลิป** และส่งผลเป็นไฟล์ `lab06_submission.csv` ซึ่งมีคอลัมน์ `id,transcript` **ตัวชี้วัด: ค่า WER รวมของทุกคลิป** (ค่ายิ่งต่ำยิ่งดี โดยไม่นับตัวพิมพ์ใหญ่-เล็กและเครื่องหมายวรรคตอน) ข้อความอ้างอิงที่ถูกต้องถูกเก็บไว้บนเซิร์ฟเวอร์ของตารางอันดับ

1. รันเซลล์ baseline ด้านล่าง ซึ่งจะสร้างไฟล์ส่งผลที่มีรูปแบบถูกต้องทันที (ประมาณ 1 นาทีบน GPU หรือหลายนาทีบน CPU)
2. หากต้องการส่งผล ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์การตั้งค่า (และรันเซลล์นั้นใหม่) จากนั้นรันเซลล์ส่งผล
3. ทดลองปรับปรุงให้ได้ผลดีกว่า baseline ในเซลล์ "ระบบที่ปรับปรุงแล้ว"

> **หลักความซื่อสัตย์ทางวิชาการ (Honour code)** ข้อความอ้างอิงที่ซ่อนไว้มาจากชุดข้อมูลสาธารณะ การค้นหาข้อความดังกล่าวจึงทำได้ง่ายและไม่เกิดประโยชน์ต่อการเรียนรู้ ตารางอันดับมีไว้เพื่อสนับสนุนการเรียนรู้ และโน้ตบุ๊ก (notebook) ของผู้เรียนจะได้รับการประเมินด้วย **ห้ามแก้ไขหรือพิมพ์ข้อความถอดความด้วยตนเอง** เนื่องจากวัตถุประสงค์คือการวัดความสามารถของ *ระบบ*

In [ ]:
# 🏆 Baseline — โมเดล Whisper 'base' ภาษาอังกฤษ ค่าตั้งต้น -> lab06_submission.csv
rows = []
for cid, f in zip(manifest["id"], manifest["file"]):
    out = model.transcribe(lab_data(f"asr/{f}"), fp16=USE_FP16, language="en")
    rows.append({"id": cid, "transcript": out["text"].strip()})
sub = pd.DataFrame(rows)
sub.to_csv("lab06_submission.csv", index=False)
print("ถอดความแล้ว", len(sub), "คลิป -> lab06_submission.csv")
display(sub.head())


In [ ]:
# 🏆 ส่งผล (เฉพาะกรณีที่ระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าแล้ว)
if NAME and JOIN_CODE:
    submit_to_leaderboard("lab06_submission.csv")
else:
    print("ข้ามการส่งผลไปยังตารางอันดับ - โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าและรันเซลล์นั้นใหม่หากต้องการส่งผล")


### แนวทางการปรับปรุงให้ดีกว่า baseline

- **โมเดลขนาดใหญ่ขึ้น:** `"small"` ให้ผลดีกว่า `"base"` อย่างชัดเจน ส่วน `"medium"` หรือ `"turbo"` สามารถรันบน GPU รุ่น T4 ได้ แต่ช้ากว่า
- **การชี้นำคำศัพท์ (vocabulary priming):** `initial_prompt="A patient describes symptoms such as headache, cough, phlegm, dizziness, rash, anemia, ligament."` ช่วยให้ Whisper ถอดศัพท์ทางการแพทย์ได้ดีขึ้น (prompt ที่ยาวหรือเฉพาะเจาะจงเกินไปอาจทำให้ผลแย่ลง จึงควรวัดผลทุกครั้ง)
- **การถอดรหัส (decoding):** `beam_size=5` (และ `best_of=5`) ค้นหาข้อความที่เป็นไปได้หลายรูปแบบมากขึ้น ส่วน `temperature=0` ทำให้ผลลัพธ์คงที่ทุกครั้ง
- **เครื่องมืออื่น:** Whisper pipeline ของ Hugging Face `transformers` หรือ `faster-whisper` รองรับการประมวลผลเป็นชุด (batching) และ checkpoint รุ่นใหม่กว่า
- **การตรวจสอบอย่างถูกต้อง:** ข้อความอ้างอิงของชุดทดสอบถูกซ่อนไว้ จึงควรสร้างชุดข้อมูลสำหรับพัฒนา (dev set) ขนาดเล็กโดยถอดความบางคลิปด้วยตนเอง (ส่วนที่ 2) แทนการปรับจูนกับตารางอันดับ ทั้งนี้ ตัวให้คะแนนแปลงเป็นตัวพิมพ์เล็กและตัดเครื่องหมายวรรคตอน แต่ "2" และ "two" ยังนับเป็นคำที่ต่างกัน

In [ ]:
# 🏆 ระบบที่ปรับปรุงแล้ว (ไม่บังคับ) - ปรับค่าที่กำหนด รันเซลล์ และรันเซลล์ส่งผลด้านบนอีกครั้ง
MY_MODEL_SIZE = "small"
MY_PROMPT     = "A patient describes symptoms such as headache, cough, phlegm, dizziness, rash, anemia, ligament."
BEAM_SIZE     = 5

better_model = models.get(MY_MODEL_SIZE) if "models" in globals() else None
better_model = better_model or whisper.load_model(MY_MODEL_SIZE)
rows = []
for cid, f in zip(manifest["id"], manifest["file"]):
    out = better_model.transcribe(lab_data(f"asr/{f}"), fp16=USE_FP16, language="en",
                                  initial_prompt=MY_PROMPT, beam_size=BEAM_SIZE, temperature=0.0)
    rows.append({"id": cid, "transcript": out["text"].strip()})
sub = pd.DataFrame(rows)
sub.to_csv("lab06_submission.csv", index=False)
print("ถอดความแล้ว", len(sub), "คลิปด้วยโมเดล", MY_MODEL_SIZE, "-> lab06_submission.csv")
display(sub.head())


## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

ให้ส่งโน้ตบุ๊กนี้ที่ **รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** ผ่านช่องทางการส่งงานของรายวิชา โดยประกอบด้วย

1. ค่า WER ของคลิปตัวอย่าง (ส่วนที่ 1) และตารางเปรียบเทียบขนาดโมเดล (ส่วนที่ 2) ปรากฏในผลลัพธ์
2. คำตอบของ ✍️ **คำถามที่ 1–4** ซึ่งรวมถึงคำอธิบาย 2–3 ประโยคเกี่ยวกับความเสี่ยงที่สำคัญที่สุดของความผิดพลาดของ ASR ในการบันทึกเวชระเบียน (โดยเฉพาะขนาดยาและการปฏิเสธ เช่น "no allergies" เทียบกับ "allergies")

*(ไม่บังคับ)* ผู้เรียนที่เข้าร่วมโจทย์ท้าทายบนตารางอันดับ 🏆 สามารถระบุชื่อที่ใช้และค่า WER ที่ดีที่สุดไว้ในส่วนนี้ได้ โดยไม่มีผลต่อการประเมินชิ้นงานที่ต้องส่ง

## การศึกษาเพิ่มเติม
- ศึกษา [Hugging Face `transformers` Whisper](https://huggingface.co/openai/whisper-small) สำหรับการประมวลผลเป็นชุด (batching) และการปรับจูน (fine-tuning) ให้เหมาะกับสำเนียงในพื้นที่
- ทดลองใช้ Whisper กับเสียงพูดภาษาไทย (`language="th"`) และพิจารณาความถูกต้องในการถอดชื่อยาและตัวเลขภาษาไทย